# QQ Plots: Checking for Normality

Many data-science models — linear regression, logistic regression, Gaussian Naive Bayes — and several statistical tests *assume the variables are Gaussian*. So "is this data normal?" is a question you will ask constantly. A histogram gives a rough first impression, but the sharper tool is the **quantile–quantile (QQ) plot**: compare your data's quantiles against a standard normal's quantiles — if the points hug the diagonal, the data is plausibly Gaussian; where and how they bend tells you exactly what is wrong. It is based on the sixteenth video of Lesson 3 (Describing Distributions).

## 1. Introduction

Checking for normality is done frequently in data science, because Gaussianity is baked into many methods as an assumption. Sometimes a quick look at a histogram settles it — the newspaper budgets clearly aren't bell-shaped. But when a histogram *does* look "a little Gaussian", how do you get a more exact answer? Comparing full curves by eye is unreliable; comparing **quantiles pointwise** is sharp.

After completing this notebook you will be able to:

- Build a QQ plot: standardize the data, compute its quantiles, compute the matching standard-normal quantiles, and scatter one against the other.
- Judge normality from how closely the points follow the reference line.
- Read the *shape* of deviations: which bend means skew, which means heavy or light tails.

## 2. Theory

### The recipe

Given a sample $x_1, \dots, x_n$, the classical QQ plot against the normal distribution:

1. **Standardize** the data: subtract the mean, divide by the standard deviation ($z_i = \frac{x_i - \bar x}{s}$).
2. Compute the **sample quantiles** — in practice, sort the standardized values; the $i$-th sorted value is the sample's $\frac{i - 0.5}{n}$ quantile.
3. Compute the **same quantiles for a standard normal** distribution — the theoretical quantiles $q_{(i-0.5)/n}$ of $N(0,1)$.
4. Draw a **scatter plot**: x-coordinates the theoretical quantiles, y-coordinates the sample quantiles. Each point corresponds to one quantile.

If the data really is normally distributed, sample and theoretical quantiles agree, so the points fall close to the **diagonal line** $y = x$.

### Reading the deviations

The QQ plot doesn't just say "no" — it says *how* the data is non-normal:

- **Points splitting away from the line at the ends**: the tails of your data disagree with normal tails. Bending *above* the line on the right = right tail too long; *below* on the left = left tail too short — together, right skew.
- **More points concentrated on the left of the plot than the right**: mass piled on low values — a sign of skew.
- An **S-shaped** curve around the line: tails heavier (or lighter) than normal on both sides.
- Points on the line except a few extremes: outliers.

### The advertising data, both ways

- **Newspaper budgets**: the histogram already looks non-Gaussian, and the QQ plot confirms it loudly — the scatter splits from the line in two marked regions, and the points bunch toward the left: skewed, not normal.
- **Sales**: the histogram is pretty bell-shaped, and its QQ plot has the quantiles aligned along the line — a Gaussian model for sales is reasonable.

## 3. Worked Examples

### Example 1: a tiny QQ plot by hand

Standardized sample of $n = 4$: sorted values $-1.3, -0.4, 0.5, 1.2$. Plotting positions $\frac{i-0.5}{4} = 0.125, 0.375, 0.625, 0.875$. Standard-normal quantiles at those levels: $-1.15, -0.32, 0.32, 1.15$. The QQ points are

$$
(-1.15, -1.3),\ (-0.32, -0.4),\ (0.32, 0.5),\ (1.15, 1.2)
$$

— all close to $y = x$: nothing in this tiny sample contradicts normality.

### Example 2: what right-skewed data does

For exponential-like data (many small values, few huge ones), after standardizing: the *largest* sample values are far bigger than normal quantiles predict (points rise above the line on the right), while the smallest values huddle just below the mean rather than stretching left (points sit above the line on the left too — a bowl/convex shape). The newspaper budgets show exactly this pattern.

### Example 3: why quantiles beat histograms here

Two histograms with 12 points can look alike for normal and slightly-skewed data — binning eats the evidence. The QQ plot uses *every point as its own quantile*, no bins, and plots against exact theory: small systematic deviations become visible as coherent bends instead of vanishing into bar heights.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project). We build the normal quantile function by inverting a numerically integrated CDF — same trick as in the quantiles notebook, no extra libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

newspaper = np.array([8.7, 14.2, 18.3, 18.4, 23.2, 25.9,
                      29.7, 35.2, 51.2, 54.7, 65.9, 75.0])

# standard-normal quantile function via CDF inversion
_grid = np.linspace(-6, 6, 200_001)
_pdf = np.exp(-0.5 * _grid**2) / np.sqrt(2 * np.pi)
_cdf = np.cumsum(_pdf) * (_grid[1] - _grid[0])

def normal_quantile(p):
    return _grid[np.searchsorted(_cdf, p)]

def qq_points(data):
    z = (data - data.mean()) / data.std()            # step 1: standardize
    sample_q = np.sort(z)                            # step 2: sample quantiles
    p = (np.arange(1, len(data) + 1) - 0.5) / len(data)
    theory_q = normal_quantile(p)                    # step 3: normal quantiles
    return theory_q, sample_q                        # step 4: scatter these

### The QQ plot of the newspaper budgets

In [ ]:
theory_q, sample_q = qq_points(newspaper)

plt.figure(figsize=(6.5, 6))
plt.plot([-2.5, 2.5], [-2.5, 2.5], color="tab:orange", linewidth=2,
         label="reference line y = x")
plt.plot(theory_q, sample_q, "o", color="tab:blue", markersize=8,
         label="quantile pairs")
plt.title("QQ plot: newspaper budgets vs. standard normal")
plt.xlabel("theoretical quantiles (standard normal)")
plt.ylabel("sample quantiles (standardized data)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("The ends split away from the line and the points bunch low/left:")
print("right-skewed, NOT Gaussian -- as the histogram already hinted.")

### A sales-like column: bell-shaped and on the line

The dataset's sales column is roughly normal; we use a normal sample with its mean and spread (about $14 \pm 5$) to reproduce the video's second QQ plot.

In [ ]:
rng = np.random.default_rng(42)
sales_like = rng.normal(14, 5, size=200)

theory_q, sample_q = qq_points(sales_like)

plt.figure(figsize=(6.5, 6))
plt.plot([-3, 3], [-3, 3], color="tab:orange", linewidth=2, label="y = x")
plt.plot(theory_q, sample_q, "o", color="tab:blue", markersize=4,
         label="quantile pairs")
plt.title("QQ plot: sales-like data vs. standard normal")
plt.xlabel("theoretical quantiles")
plt.ylabel("sample quantiles")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("Quantiles aligned along the line -> a Gaussian model is reasonable here.")

### Histograms vs. QQ plots, side by side

The video's opening comparison: histogram intuition on the left, the sharper QQ verdict on the right, for both columns.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for row, (data, name) in enumerate([(newspaper, "newspaper (12 budgets)"),
                                    (sales_like, "sales-like (200 values)")]):
    axes[row, 0].hist(data, bins=10 if len(data) < 50 else 25, density=True,
                      color="tab:blue", alpha=0.7, edgecolor="white")
    axes[row, 0].set_title(f"{name}: histogram")
    axes[row, 0].set_ylabel("density")

    tq, sq = qq_points(data)
    lim = 3
    axes[row, 1].plot([-lim, lim], [-lim, lim], color="tab:orange", linewidth=2)
    axes[row, 1].plot(tq, sq, "o", color="tab:blue", markersize=4)
    axes[row, 1].set_title(f"{name}: QQ plot")
    axes[row, 1].set_xlabel("theoretical quantiles")
    axes[row, 1].set_ylabel("sample quantiles")
for ax in axes.flat:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Visualizations

### A field guide to QQ shapes

Four synthetic datasets and their QQ signatures — the patterns worth memorizing.

In [ ]:
rng = np.random.default_rng(42)

n = 500
datasets = [
    ("normal: on the line", rng.normal(0, 1, n)),
    ("right-skewed: bowl shape", rng.exponential(1, n)),
    ("heavy tails: S through the line", rng.standard_t(3, n)),
    ("light tails (uniform): S the other way", rng.uniform(0, 1, n)),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, (name, data) in zip(axes, datasets):
    tq, sq = qq_points(data)
    ax.plot([-3.4, 3.4], [-3.4, 3.4], color="tab:orange", linewidth=2)
    ax.plot(tq, sq, "o", color="tab:blue", markersize=3)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("theoretical quantiles")
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-4.5, 4.5)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("sample quantiles")
plt.suptitle("How to read a QQ plot: each departure from normality has a signature")
plt.tight_layout()
plt.show()

### Sampling noise: how straight is straight enough?

Even genuinely normal samples wobble around the line, especially at the tails and for small $n$. Eight QQ plots of *true* normal data, $n = 20$ each — calibrate your eye before declaring non-normality.

In [ ]:
rng = np.random.default_rng(7)

fig, axes = plt.subplots(2, 4, figsize=(14, 6.5), sharex=True, sharey=True)
for ax in axes.flat:
    tq, sq = qq_points(rng.normal(0, 1, 20))
    ax.plot([-2.6, 2.6], [-2.6, 2.6], color="tab:orange", linewidth=1.5)
    ax.plot(tq, sq, "o", color="tab:blue", markersize=4)
    ax.grid(True, alpha=0.3)
fig.suptitle("Eight QQ plots of TRULY normal samples (n = 20): tail wobble is normal")
fig.supxlabel("theoretical quantiles")
fig.supylabel("sample quantiles")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Do the Example-1 hand computation in code: for the standardized sorted sample $[-1.3, -0.4, 0.5, 1.2]$, compute the plotting positions $\frac{i-0.5}{4}$ and the corresponding normal quantiles, and confirm the four QQ points.

**Exercise 2.** Draw the QQ plot of 300 samples from a uniform distribution. Which way does the "S" bend, and what does that say about the uniform's tails compared to the normal's?

### Intermediate

**Exercise 3.** The die-roll QQ plot: make a QQ plot of 500 fair die rolls. Explain the staircase pattern — what feature of the data produces horizontal runs of points?

**Exercise 4.** Log transforms often "fix" right skew: draw QQ plots of lognormal data (`rng.lognormal(0, 0.5, 500)`) before and after taking `np.log`. What happens to the bowl shape, and why does this matter for the models listed in the introduction?

### Advanced

**Exercise 5.** Sensitivity vs. the histogram: generate 100 samples from a *mildly* skewed distribution (e.g. `rng.gamma(8, 1, 100)`), plot its histogram (which looks bell-ish) and its QQ plot. Which tool exposes the skew more clearly? Quantify the skewness value to referee.

**Exercise 6.** A QQ plot need not compare against the normal: build a QQ plot of exponential data against *exponential* theoretical quantiles ($q_p = -\ln(1-p)$ for rate 1, standardize both sides). Confirm the points now sit on the line — the same data that failed the normal QQ passes its own.

## 7. Solutions

In [ ]:
# Exercise 1: the hand computation in code
sample = np.array([-1.3, -0.4, 0.5, 1.2])
p = (np.arange(1, 5) - 0.5) / 4
tq = normal_quantile(p)
print("Exercise 1:")
print(f"plotting positions: {p}")
print(f"normal quantiles:   {np.round(tq, 2)}")
print("QQ points:", list(zip(np.round(tq, 2), sample)))

In [ ]:
# Exercise 2: uniform data
rng = np.random.default_rng(0)
tq, sq = qq_points(rng.uniform(0, 1, 300))

plt.figure(figsize=(5.5, 5))
plt.plot([-3, 3], [-3, 3], color="tab:orange", linewidth=2)
plt.plot(tq, sq, "o", markersize=3)
plt.title("Exercise 2: uniform data vs normal quantiles")
plt.xlabel("theoretical"); plt.ylabel("sample")
plt.grid(True, alpha=0.3)
plt.show()

print("The S bends flat at both ends: extreme sample quantiles are LESS extreme")
print("than normal theory expects. The uniform has hard edges -- lighter tails")
print("than the normal, which stretches to infinity.")

In [ ]:
# Exercise 3: die rolls -- a discrete staircase
rng = np.random.default_rng(0)
tq, sq = qq_points(rng.integers(1, 7, 500).astype(float))

plt.figure(figsize=(5.5, 5))
plt.plot([-3, 3], [-3, 3], color="tab:orange", linewidth=2)
plt.plot(tq, sq, "o", markersize=3)
plt.title("Exercise 3: 500 die rolls")
plt.xlabel("theoretical"); plt.ylabel("sample")
plt.grid(True, alpha=0.3)
plt.show()

print("Only six distinct values exist, so sorted data forms six flat runs --")
print("horizontal steps in the plot. QQ plots reveal discreteness instantly;")
print("no continuous Gaussian can match a staircase.")

In [ ]:
# Exercise 4: log-transforming skewed data
rng = np.random.default_rng(0)
lognorm = rng.lognormal(0, 0.5, 500)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, data, name in [(axes[0], lognorm, "lognormal (raw): bowl = right skew"),
                       (axes[1], np.log(lognorm), "after np.log: straight line")]:
    tq, sq = qq_points(data)
    ax.plot([-3, 3], [-3, 3], color="tab:orange", linewidth=2)
    ax.plot(tq, sq, "o", markersize=3)
    ax.set_title(name)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Exercise 4: the log undoes the multiplicative skew -- log(lognormal) IS")
print("normal, so the bowl straightens. That is why practitioners log-transform")
print("skewed features before fitting Gaussian-assuming models.")

In [ ]:
# Exercise 5: QQ beats histogram on mild skew
rng = np.random.default_rng(0)
gamma_data = rng.gamma(8, 1, 100)
z = (gamma_data - gamma_data.mean()) / gamma_data.std()
skew = np.mean(z**3)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].hist(gamma_data, bins=15, density=True, color="tab:blue", alpha=0.7,
             edgecolor="white")
axes[0].set_title("histogram: looks bell-ish")
tq, sq = qq_points(gamma_data)
axes[1].plot([-3, 3], [-3, 3], color="tab:orange", linewidth=2)
axes[1].plot(tq, sq, "o", markersize=4)
axes[1].set_title("QQ plot: coherent bend visible")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Exercise 5: skewness = {skew:.2f} (mild but real). The histogram's 15 bars")
print("hide it; the QQ plot shows a consistent bowl-shaped drift off the line,")
print("because every point contributes as its own quantile -- no binning.")

In [ ]:
# Exercise 6: QQ against the RIGHT theory
rng = np.random.default_rng(0)
expo = rng.exponential(1, 500)

z = np.sort((expo - expo.mean()) / expo.std())
p = (np.arange(1, 501) - 0.5) / 500
theo = -np.log(1 - p)                          # exponential(1) quantiles
theo = (theo - theo.mean()) / theo.std()       # standardized the same way

plt.figure(figsize=(5.5, 5))
plt.plot([-2, 5], [-2, 5], color="tab:orange", linewidth=2)
plt.plot(theo, z, "o", markersize=3)
plt.title("Exercise 6: exponential data vs exponential quantiles")
plt.xlabel("theoretical (exponential)"); plt.ylabel("sample")
plt.grid(True, alpha=0.3)
plt.show()

print("Exercise 6: on the line. A QQ plot tests agreement with WHICHEVER")
print("distribution supplies the theoretical quantiles -- the normal is just the")
print("most common choice.")

## 8. Conclusion

- Normality checks matter because many models (linear/logistic regression, Gaussian Naive Bayes) and tests **assume Gaussian variables**.
- A **QQ plot** compares data against theory quantile by quantile: standardize, sort (sample quantiles), compute matching standard-normal quantiles, scatter theory (x) vs. sample (y). Normal data hugs the line $y = x$.
- Deviations are diagnostic, not just binary: end-splitting and left-bunching = **skew** (the newspaper budgets); an S through the line = heavy tails; a flattened S = light tails; staircases = discreteness.
- The aligned QQ plot of the sales column is what "plausibly Gaussian" looks like — and small-sample tail wobble is expected even for truly normal data, so calibrate before over-reading.
- The idea generalizes: supply any distribution's quantiles as the theory axis and the QQ plot tests *that* distribution instead.

With QQ plots the lesson's toolkit is complete — numbers (moments, quantiles) and pictures (box, KDE, violin, QQ) for describing one distribution. Next: **joint distributions** of several variables.